# 🔧 題目 6：不動產房價趨勢 — Solution

⚠️ 講師用。學員請用 `pipeline_starter.ipynb`。


## Section 0：環境設定


In [ ]:
import pandas as pd
import sqlite3
import os
import json
print('✅ 套件載入完成')


In [ ]:
OPENAI_API_KEY = ""
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]
print("✅" if OPENAI_API_KEY else "⚠️ fallback")


---
## Section 1：Extract


### Step 1-1：讀取 CSV


In [ ]:
df_raw = pd.read_csv("real_estate.csv")
print(f"📊 {len(df_raw)} 筆, {len(df_raw.columns)} 欄")
df_raw.head()


### Step 1-2：檢查


In [ ]:
print(df_raw.dtypes)
print("\n", df_raw.isnull().sum())
print("\n", df_raw.describe())


### Step 1-3：自由探索


In [ ]:
print(df_raw.iloc[:, 0].value_counts().head(10))
print(f"\n唯一值: {df_raw.iloc[:, 0].nunique()}")


### Step 1-4：SQLite


In [ ]:
conn = sqlite3.connect("pipeline.db")
df_raw.to_sql("raw_realestate", conn, if_exists="replace", index=False)
print(f"✅ raw_realestate: {pd.read_sql('SELECT COUNT(*) as n FROM raw_realestate', conn)['n'][0]} 筆")


---
## Section 2：Transform


### Step 2-1：從 raw 讀出


In [ ]:
df = pd.read_sql("SELECT * FROM raw_realestate", conn)
before = len(df)
print(f"讀出 {before} 筆")


### Step 2-2 ~ 2-4：清洗


In [ ]:
df = df.dropna(subset=["總價元", "鄉鎮市區"])
df["總價元"] = pd.to_numeric(df["總價元"], errors="coerce")
df["單價元平方公尺"] = pd.to_numeric(df["單價元平方公尺"], errors="coerce")
df["建物移轉總面積平方公尺"] = pd.to_numeric(df["建物移轉總面積平方公尺"], errors="coerce")
df = df[df["總價元"] > 0]
df["總價萬"] = (df["總價元"] / 10000).round(1)
df["面積坪"] = (df["建物移轉總面積平方公尺"] / 3.30579).round(1)
# 民國年轉西元年
def roc_to_ad(d):
    try:
        s = str(int(d))
        y = int(s[:-4]) + 1911
        m = int(s[-4:-2])
        return f"{y}-{m:02d}"
    except: return None
df["交易年月"] = df["交易年月日"].apply(roc_to_ad)


In [ ]:
print(f"清洗前: {before} → 清洗後: {len(df)}")


### 🏁 檢查點


In [ ]:
assert (df["總價元"] > 0).all(), "❌ 總價有非正值"
assert "總價萬" in df.columns, "❌ 缺少總價萬"
assert "面積坪" in df.columns, "❌ 缺少面積坪"
print("✅ 通過")
print(f"   {len(df)} 筆, {len(df.columns)} 欄")


### Step 2-5：寫入 cleaned


In [ ]:
df.to_sql("cleaned_realestate", conn, if_exists="replace", index=False)
print(f"✅ cleaned_realestate")


---
## Section 3：SQL


### Step 3-1：各縣市均價


In [ ]:
city_stats = pd.read_sql("""
SELECT 縣市,
       COUNT(*) as 交易量,
       ROUND(AVG(總價萬), 1) as 平均總價萬,
       ROUND(AVG(面積坪), 1) as 平均坪數
FROM cleaned_realestate
GROUP BY 縣市
ORDER BY 平均總價萬 DESC
""", conn)
city_stats


### Step 3-2：建物型態分佈


In [ ]:
type_stats = pd.read_sql("""
SELECT 建物型態, COUNT(*) as 數量,
       ROUND(AVG(總價萬), 1) as 平均總價萬
FROM cleaned_realestate
WHERE 建物型態 IS NOT NULL AND 建物型態 != ''
GROUP BY 建物型態
ORDER BY 數量 DESC
""", conn)
type_stats


### Step 3-3：視覺化


In [ ]:
import matplotlib.pyplot as plt
city_stats.head(10).plot.barh(x=city_stats.columns[0], y=city_stats.columns[-1], figsize=(10,5))
plt.tight_layout()
plt.show()


### Step 3-5：存結果


In [ ]:
os.makedirs("processed", exist_ok=True)
city_stats.to_csv("processed/city_stats.csv", index=False)
type_stats.to_csv("processed/type_stats.csv", index=False)
print("✅ 已存")


---
## Section 4：LLM


In [ ]:
import requests
def llm_analyze(text, api_key=None):
    if api_key: return _llm_api(text, api_key)
    return _llm_fallback(text)
def _llm_api(text, api_key):
    prompt = f"""請分析以下台灣不動產區域，回傳 JSON：
{{"area_character": "蛋黃區/蛋白區/郊區/新興區/其他", "insight": "一句話區域分析"}}\n文字：{text[:300]}"""
    try:
        resp = requests.post("https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3}, timeout=30)
        content = resp.json()["choices"][0]["message"]["content"].strip()
        start = content.find("{")
        end = content.rfind("}")
        if start != -1 and end != -1:
            content = content[start:end + 1]
        return json.loads(content)
    except: return _llm_fallback(text)
def _llm_fallback(text):
    t = text
    if any(w in t for w in ["大安","信義","中正","松山","中山"]): cat = "蛋黃區"
    elif any(w in t for w in ["內湖","南港","士林","北投","文山"]): cat = "蛋白區"
    elif any(w in t for w in ["淡水","三峽","鶯歌","林口","五股"]): cat = "郊區"
    elif any(w in t for w in ["青埔","竹北","新莊","板橋"]): cat = "新興區"
    else: cat = "其他"
    return {"area_character": cat, "insight": text[:30] + "..."}
print("✅ LLM Helper")


### Step 4-1：單筆測試


In [ ]:
test = str(df["鄉鎮市區"].iloc[0])
result = llm_analyze(test, OPENAI_API_KEY if OPENAI_API_KEY else None)
print(f"📝 {test[:60]}\n🤖 {result}")


### Step 4-2：批次


In [ ]:
BATCH_SIZE = 50
api_key = OPENAI_API_KEY if OPENAI_API_KEY else None
results = []
for i, row in df.head(BATCH_SIZE).iterrows():
    r = llm_analyze(str(row["鄉鎮市區"]), api_key)
    results.append(r)
    if len(results) % 10 == 0: print(f"  {len(results)}/{BATCH_SIZE}")
print(f"✅ {len(results)} 筆")


### Step 4-3：整理 + 寫入


In [ ]:
df_analyzed = df.head(BATCH_SIZE).copy()
first_keys = list(results[0].keys())
for k in first_keys:
    df_analyzed[k] = [r.get(k, "") for r in results]
df_analyzed.rename(columns={k: "llm_insight" for k in first_keys if "insight" in k}, inplace=True)
df_analyzed.to_sql("analyzed_realestate", conn, if_exists="replace", index=False)
print("📊 三表：")
for t in ["raw_realestate", "cleaned_realestate", "analyzed_realestate"]:
    print(f"  {t}: {pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', conn)['n'][0]}")


---
## Section 5：驗證


In [ ]:
lineage = pd.read_sql("""
SELECT \'raw_realestate\' as layer, COUNT(*) as rows FROM raw_realestate
UNION ALL SELECT \'cleaned_realestate\', COUNT(*) FROM cleaned_realestate
UNION ALL SELECT \'analyzed_realestate\', COUNT(*) FROM analyzed_realestate
""", conn)
print(lineage.to_string(index=False))


---
## Section 6：報告


In [ ]:
avg_price = df["總價萬"].mean()
city_lines = "\n".join(
    f'- {r["縣市"]}: {r["平均總價萬"]} 萬（{r["交易量"]} 筆，平均 {r["平均坪數"]} 坪）'
    for _, r in city_stats.iterrows()
)
report = f"""# 不動產房價趨勢分析報告
## 資料概要
- 分析交易：{len(df)} 筆
- 平均總價：{avg_price:.1f} 萬元
## 各縣市均價
{city_lines}
## 建議
1. 蛋黃區價格穩定適合保值
2. 新興區有成長空間
## Pipeline
CSV → pandas → SQLite → SQL → LLM → 本報告
"""
os.makedirs("output", exist_ok=True)
with open("output/pipeline_doc.md", "w") as f: f.write(report)
print("✅ pipeline_doc.md")


---
## Section 7：打包


In [ ]:
checks = [("pipeline.db","DB"), ("processed","統計"), ("output/pipeline_doc.md","報告")]
for p,d in checks: print(f"  {'✅' if os.path.exists(p) else '❌'} {d}: {p}")
c = sqlite3.connect("pipeline.db")
for t in ["raw_realestate","cleaned_realestate","analyzed_realestate"]:
    try: print(f"  ✅ {t}: {pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', c)['n'][0]}")
    except: print(f"  ❌ {t}")
c.close()


---
## Section 8：FastAPI


In [ ]:
from fastapi import FastAPI, HTTPException
import os

api = FastAPI(title='不動產房價趨勢 API')
DB_PATH = "pipeline.db"

def get_conn():
    if not os.path.exists(DB_PATH):
        raise HTTPException(status_code=500, detail="pipeline.db 不存在，請先完成前面 pipeline")
    return sqlite3.connect(DB_PATH)

@api.get("/health")
def health():
    c = get_conn()
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", c)
    c.close()
    return {"status": "ok", "tables": tables["name"].tolist()}

@api.get("/stats")
def stats():
    c = get_conn()
    df_api = pd.read_sql("""SELECT 縣市, COUNT(*) as 交易量, ROUND(AVG(總價萬),1) as 均價萬 FROM cleaned_realestate GROUP BY 縣市 ORDER BY 均價萬 DESC""", c)
    c.close()
    return df_api.to_dict(orient="records")

@api.get("/analyzed")
def analyzed(limit: int = 20):
    c = get_conn()
    safe_limit = min(limit, 100)
    df_api = pd.read_sql(f"""SELECT 鄉鎮市區, area_character, llm_insight FROM analyzed_realestate LIMIT {limit}""", c)
    c.close()
    return df_api.to_dict(orient="records")

@api.get("/summary")
def summary():
    c = get_conn()
    result = {}
    for table in ['raw_realestate', 'cleaned_realestate', 'analyzed_realestate']:
        try:
            result[table] = int(pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", c)["n"][0])
        except Exception:
            result[table] = 0
    c.close()
    return result

@api.get("/report")
def report():
    path = "output/pipeline_doc.md"
    if not os.path.exists(path):
        raise HTTPException(status_code=404, detail="pipeline_doc.md 不存在")
    with open(path) as f:
        return {"report": f.read()}

print("✅ API 定義完成")
print("📡 /health:", health())
print("📡 /stats:", stats()[:3])
print("📡 /analyzed:", analyzed()[:3])
print("📡 /summary:", summary())


In [ ]:
# 啟動 API + 測試
import subprocess, sys, time, requests as req, os

PORT = 8000
if os.path.exists('api.py'):
    proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'api:app',
         '--host', '127.0.0.1', '--port', str(PORT), '--log-level', 'warning'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(20):
        try:
            req.get(f'http://127.0.0.1:{PORT}/health', timeout=1); break
        except: time.sleep(0.5)
    else: print('❌ 啟動失敗')

print('📡 /health:', req.get(f'http://127.0.0.1:{PORT}/health').json())
print('📡 /stats:', req.get(f'http://127.0.0.1:{PORT}/stats').json()[:3])
print('📡 /analyzed:', req.get(f'http://127.0.0.1:{PORT}/analyzed').json()[:3])


> 完整版在 `api.py`。本地啟動請用 Terminal：`uvicorn api:app --reload --port 8000`


---
## Section 9：Dashboard


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

c = sqlite3.connect("pipeline.db")
stats = pd.read_sql("""SELECT 縣市, COUNT(*) as 交易量, ROUND(AVG(總價萬),1) as 均價萬 FROM cleaned_realestate GROUP BY 縣市 ORDER BY 均價萬 DESC""", c)
analyzed_preview = pd.read_sql("SELECT * FROM analyzed_realestate LIMIT 20", c)
c.close()

view = widgets.ToggleButtons(
    options=[("統計", "stats"), ("LLM", "llm")],
    description="資料："
)

def update_dashboard(tab):
    clear_output(wait=True)
    display(view)
    if tab == "stats":
        display(stats.head(20))
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        stats.set_index("縣市")["均價萬"].sort_values().plot.barh(ax=ax, color="teal")
        ax.set_title("各縣市均價")
        plt.tight_layout()
        plt.show()
    else:
        display(analyzed_preview)

widgets.interact(update_dashboard, tab=view)


> 完整版在 `app.py`。本地啟動請用 Terminal：`streamlit run app.py`


---
## Section 10：本地部署指引

完整版已放在同資料夾的 `api.py` 與 `app.py`。在兩個 Terminal 分別執行：

```bash
cd data/raw/topic_6
uvicorn api:app --reload --port 8000
```

```bash
cd data/raw/topic_6
streamlit run app.py
```

API 啟動後可測：
- `http://127.0.0.1:8000/health`
- `http://127.0.0.1:8000/stats`
- `http://127.0.0.1:8000/analyzed`
- `http://127.0.0.1:8000/report`
